# YOLO caries training: AI-written DRAFT (reference only)

**This notebook was written by Claude (AI) as a reference, not as the submitted work.** Write your own `02_train.ipynb`; use this only to see what the steps are. Logged in the AI Use Log.

What it does:
1. Trains YOLO on `data/processed/yolo/` (made by `scripts/Junwei/dedup_and_split.py`, split by person)
2. Scores the best checkpoint on the **test** split (never seen during training)
3. Appends one row to `results/experiments.csv`
4. Shows the training curves and test plots

The same code produced the feasibility results in `results/Junwei/ai_feasibility/feasibility_ai/` (test mAP50 0.924).

## 1. Setup

In [ ]:
from pathlib import Path
import csv
import datetime

import yaml
from IPython.display import Image, display
from ultralytics import YOLO

REPO = Path.cwd().parents[2]                                   # notebooks -> yolo_caries -> models -> repo root
DATA = REPO / "models/yolo_caries/configs/data.yaml"
RUNS = REPO / "runs/yolo_caries"                               # gitignored; Ultralytics writes each run here
LOG = REPO / "models/yolo_caries/results/experiments.csv"
assert DATA.exists(), f"path not found: {DATA}"

## 2. Settings

Change only this cell between experiments, and give each run a new `RUN_NAME`.

In [ ]:
MODEL = "yolo11n.pt"        # starting weights (pre-trained on COCO); try yolo11s.pt for a bigger model
RUN_NAME = "draft_baseline"
EPOCHS = 100
IMGSZ = 640                 # images are resized to this; larger helps small lesions but is slower
PATIENCE = 20               # stop early if val mAP has not improved for this many epochs
WORKERS = 4                 # 8 ran out of laptop RAM (cuDNN HOST_ALLOCATION_FAILED); use 2 if it happens again

## 3. Train

Uses the train split to learn and the val split to pick the best epoch (`best.pt`). Prints `CUDA:0 (NVIDIA GeForce RTX 4050 ...)` at the start if the GPU is used: if it says `CPU`, the wrong Python environment is active.

In [ ]:
model = YOLO(MODEL)
model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, patience=PATIENCE, workers=WORKERS,
            project=RUNS, name=RUN_NAME, exist_ok=True)

## 4. Test

The test split is used once, here, to report the final score. Never use it to choose settings: that is what val is for.

In [ ]:
run_dir = RUNS / RUN_NAME
best = YOLO(run_dir / "weights" / "best.pt")
m = best.val(data=DATA, split="test", workers=WORKERS, project=RUNS, name=f"{RUN_NAME}_test", exist_ok=True)
scores = {"mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
          "precision": float(m.box.mp), "recall": float(m.box.mr)}
{k: round(v, 3) for k, v in scores.items()}

## 5. Log the run

Appends one row to `results/experiments.csv`, reading batch and lr0 from what Ultralytics actually used (`args.yaml`).

In [ ]:
args = yaml.safe_load((run_dir / "args.yaml").read_text())
epochs_run = sum(1 for _ in open(run_dir / "results.csv")) - 1   # fewer than EPOCHS if early stopping kicked in
row = {
    "run_id": RUN_NAME, "date": datetime.date.today().isoformat(), "member": "ang_jun_wei",
    "model": MODEL, "data_version": "manifests/yolo_split.csv (person groups)",
    "imgsz": IMGSZ, "epochs": epochs_run, "batch": args["batch"], "lr0": args["lr0"],
    **{k: round(v, 3) for k, v in scores.items()},
    "weights_path": str((run_dir / "weights" / "best.pt").relative_to(REPO)), "notes": "",
}
with open(LOG, newline="") as f:
    header = next(csv.reader(f))
with open(LOG, "a", newline="") as f:
    csv.DictWriter(f, fieldnames=header).writerow(row)
row

## 6. Look at the results

- `results.png`: losses should fall and mAP should rise then flatten. Val loss rising while train loss falls means overfitting.
- `BoxPR_curve.png`: precision vs recall on test; the area under it is mAP50.
- `val_batch0_pred.jpg` vs `val_batch0_labels.jpg`: the model's boxes next to the annotators' boxes. Look for missed lesions and false alarms.

In [ ]:
display(Image(run_dir / "results.png", width=900))
test_dir = RUNS / f"{RUN_NAME}_test"
display(Image(test_dir / "BoxPR_curve.png", width=600))
display(Image(test_dir / "val_batch0_labels.jpg", width=900))
display(Image(test_dir / "val_batch0_pred.jpg", width=900))